# Data Acquisition

In this notebook, we will collect the data that will be used throughout the project. The process is as follows:

1. Set up the project and verify access to CollegeFootballData (CFBD)
2. Download games, team box scores, advanced statistics, betting lines, talent, and recruiting data
3. Cache the original responses and save normalized tables
4. Check the coverage of each table

The model will only use games where both participants were FBS teams. Betting data is downloaded here for later comparison, but it will not be included in the statistics-only model.

In [1]:
from pathlib import Path
import sys

# Allow the notebook to run from either the project root or the notebooks folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Imports and Global Variables

The free Academic CFBD tier is sufficient for this project. Historical responses are cached, so they will not be downloaded again unless the current season is refreshed.

In [2]:
import pandas as pd

from src.config import CURRENT_SEASON, START_SEASON, create_project_directories
from src.data import download_all_data, load_api_key

pd.set_option('display.max_columns', 100)
create_project_directories()

# The API key is read from .env. The value itself is never printed.
api_key = load_api_key([PROJECT_ROOT / '.env', PROJECT_ROOT.parent / '.env'])
print('CFBD API key found.')

CFBD API key found.


## Downloading the Data

We will use the 2014 season as the first season because reliable play-level advanced statistics are available from this point forward. The current season is refreshed each time this cell runs, while prior seasons are loaded from the local cache.

In [3]:
tables = download_all_data(
    start_season=START_SEASON,
    end_season=CURRENT_SEASON,
    refresh_current=True,
)

## Initial Data Review

In [4]:
coverage = []
for table_name, dataframe in tables.items():
    coverage.append({
        'table': table_name,
        'records': len(dataframe),
        'features': len(dataframe.columns),
    })

coverage_df = pd.DataFrame(coverage).sort_values('table')
coverage_df

,table,records,features
2,advanced_stats,26680,61
0,games,11262,42
3,lines,13800,4
5,recruiting,2790,4
4,talent,2413,3
1,team_stats,21258,45


In [5]:
games = tables['games']
fbs_games = games[
    games['home_classification'].eq('fbs') &
    games['away_classification'].eq('fbs')
]

games_by_season = (
    fbs_games.groupby('season')
    .agg(games=('game_id', 'size'), completed=('completed', 'sum'))
    .reset_index()
)
games_by_season

,season,games,completed
0,2014,760,760
1,2015,765,765
2,2016,760,760
3,2017,776,776
4,2018,772,772
5,2019,774,774
6,2020,534,534
7,2021,770,770
8,2022,776,776
9,2023,792,792


## Conclusion

At this point, the original API responses have been cached in `data/raw`, and six normalized tables have been saved in `data/interim`. The next notebook will clean these tables, construct a team-game dataset, and explore the historical data.